# CS781 Phase 6 — A1 / FULL

**Standalone Kaggle notebook.** Uses your original Phase 1 **A1** backbone and the official Phase 6 dataset. Every action has its own short, commented cell. **Run from top to bottom with GPU + Internet enabled.**

It discovers your Hugging Face username from the Kaggle secret named `HF`, fetches your original Phase 1 checkpoint and matching tokenizer, trains, predicts, validates, and **uploads the entire experiment workspace** to a new private repository. The model is not downloaded or trained until you execute the notebook in Kaggle. No tokens are stored in the output files.

The official metric is BERTScore F1; local ROUGE evaluation is used by default to avoid downloading an unrelated pretrained scoring model. Real training, remote backup, and Codabench submission have not been executed here.


**Phase 5 continuity:** source-file names, standalone `%%writefile` cells, `--run` prediction, `--pred` validation, run folders, private HF backups, and fail-fast checkpoint/ID checks follow your Phase 5 workflow. Generation-related math is necessarily different.


*Full-workspace HF uploads are handled only by `phase6_hf.py` to avoid duplicate upload logic.*


In [ ]:
# 1. Install compatible dependencies (avoids the Kaggle Hub/Gradio/Diffusers conflict).
# Transformers v5 needs Hugging Face Hub >=1; Pillow<13 is required by Gradio.
%pip -q install "transformers>=5.0,<6" "huggingface_hub>=1.32,<2" "pillow>=8,<13" "safetensors>=0.4" "tokenizers>=0.21" hf_xet "bert-score==0.3.13"


In [ ]:
# 1b. Check installed versions and the tokenizer API before training.
# This catches incompatible Kaggle package combinations early.
import importlib.metadata as versions
from transformers import PreTrainedTokenizerFast
from transformers.modeling_outputs import CausalLMOutput, Seq2SeqLMOutput
from huggingface_hub import HfApi
for name in ('transformers', 'huggingface_hub', 'pillow'):
    print(f'{name}: {versions.version(name)}')
print('Package/API preflight passed.')


In [ ]:
# 2. Give this experiment its own Kaggle workspace.
import os, sys, subprocess
from pathlib import Path
ROOT = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
WORKDIR = ROOT / 'phase6_a1_full'
(WORKDIR / 'phase6_backbones').mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Workspace:', WORKDIR)

## Shared Python source files — same organisation as Phase 5

`phase6_core.py` → `phase6_train.py` → `phase6_predict.py` → `phase6_validate.py`.
`phase6_hf.py` performs private checkpoint recovery and full-workspace backup.
Phase 6 uses *summary generation* instead of Phase 5 multi-label classification.
Each `%%writefile` cell creates one module; you do not need to upload Python files separately.

In [ ]:
%%writefile phase6_backbones/__init__.py
"""Original Phase 1 model modules: loaded only when selected."""

In [ ]:
%%writefile phase6_backbones/a1.py
# VERBATIM ORIGINAL PHASE-1 MODEL CLASS DEFINITIONS (extracted from architetcture-1.ipynb).
# Original classes are not modified; only the notebook model-instantiation code is omitted.
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers.modeling_outputs import CausalLMOutput


# ============================================================
# ARCHITECTURE 1:
# CUSTOM DECODER-ONLY TRANSFORMER FROM SCRATCH
# ============================================================


class DecoderBlock(nn.Module):
    """
    One Transformer decoder block.

    Structure:
        LayerNorm
        -> Causal Multi-Head Self Attention
        -> Residual Connection
        -> LayerNorm
        -> Feed Forward Network
        -> Residual Connection
    """

    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        # Normalize before self-attention.
        self.norm1 = nn.LayerNorm(
            d_model
        )

        # Multi-head self-attention.
        #
        # Query, Key and Value all come from
        # the same token sequence.
        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True
        )

        # Normalize before feed-forward network.
        self.norm2 = nn.LayerNorm(
            d_model
        )

        # Transformer feed-forward network.
        self.feed_forward = nn.Sequential(
            nn.Linear(
                d_model,
                d_ff
            ),
            nn.GELU(),
            nn.Linear(
                d_ff,
                d_model
            )
        )

        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        x,
        causal_mask
    ):

        # ----------------------------------------------------
        # 1. Causal self-attention
        # ----------------------------------------------------

        z = self.norm1(x)

        attention_output, _ = self.attention(
            z,
            z,
            z,
            attn_mask=causal_mask,
            need_weights=False
        )

        # Residual connection.
        x = x + self.dropout(
            attention_output
        )

        # ----------------------------------------------------
        # 2. Feed-forward network
        # ----------------------------------------------------

        z = self.norm2(x)

        ff_output = self.feed_forward(
            z
        )

        # Second residual connection.
        x = x + self.dropout(
            ff_output
        )

        return x



class DecoderTransformerLM(nn.Module):
    """
    Complete custom decoder-only Transformer
    language model.
    """

    def __init__(
        self,
        vocab_size,
        context_length=2048,
        d_model=640,
        n_heads=10,
        n_layers=8,
        d_ff=2560,
        dropout=0.1
    ):
        super().__init__()

        self.context_length = (
            context_length
        )

        self.vocab_size = (
            vocab_size
        )

        # Safety check:
        # embedding dimension must divide evenly
        # across attention heads.
        assert (
            d_model % n_heads == 0
        ), (
            "d_model must be divisible "
            "by n_heads"
        )

        # ----------------------------------------------------
        # Token embeddings
        # ----------------------------------------------------

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        # ----------------------------------------------------
        # Learned positional embeddings
        # ----------------------------------------------------

        self.position_embedding = nn.Embedding(
            context_length,
            d_model
        )

        # ----------------------------------------------------
        # Stack of custom Transformer decoder blocks
        # ----------------------------------------------------

        self.blocks = nn.ModuleList([
            DecoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                d_ff=d_ff,
                dropout=dropout
            )
            for _ in range(n_layers)
        ])

        # Final normalization.
        self.final_norm = nn.LayerNorm(
            d_model
        )

        # Convert hidden representation
        # into vocabulary logits.
        self.output_layer = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )


    def forward(
        self,
        input_ids,
        labels=None,
        attention_mask=None,
        **kwargs
    ):

        _, seq_len = input_ids.shape

        if seq_len > self.context_length:
            raise ValueError(
                f"Sequence length {seq_len} exceeds "
                f"context length {self.context_length}"
            )

        # ----------------------------------------------------
        # 1. Token embeddings
        # ----------------------------------------------------

        token_vectors = (
            self.token_embedding(
                input_ids
            )
        )

        # ----------------------------------------------------
        # 2. Positional embeddings
        # ----------------------------------------------------

        positions = torch.arange(
            seq_len,
            device=input_ids.device
        )

        position_vectors = (
            self.position_embedding(
                positions
            )
        )

        x = (
            token_vectors
            +
            position_vectors
        )

        # ----------------------------------------------------
        # 3. Causal mask
        #
        # True = attention blocked.
        #
        # Token i cannot see tokens after i.
        # ----------------------------------------------------

        causal_mask = torch.triu(
            torch.ones(
                seq_len,
                seq_len,
                device=input_ids.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        # ----------------------------------------------------
        # 4. Transformer blocks
        # ----------------------------------------------------

        for block in self.blocks:

            x = block(
                x,
                causal_mask
            )

        # ----------------------------------------------------
        # 5. Vocabulary prediction
        # ----------------------------------------------------

        x = self.final_norm(
            x
        )

        logits = self.output_layer(
            x
        )

        loss = None

        # ----------------------------------------------------
        # 6. Next-token language modelling loss
        # ----------------------------------------------------

        if labels is not None:

            # Example:
            #
            # logits:
            # t0 t1 t2
            #
            # targets:
            # t1 t2 t3

            shift_logits = (
                logits[:, :-1, :]
                .contiguous()
            )

            shift_labels = (
                labels[:, 1:]
                .contiguous()
            )

            loss = F.cross_entropy(
                shift_logits.view(
                    -1,
                    shift_logits.size(-1)
                ),
                shift_labels.view(-1)
            )

        return CausalLMOutput(
            loss=loss,
            logits=logits
        )


# ============================================================
# CREATE OUR CUSTOM MODEL
# ============================================================


In [ ]:
%%writefile phase6_core.py
"""Phase 6 shared models/data helpers — the Phase 5 core.py pattern.
The original Phase-1 architectures live in phase6_backbones/ and load strictly.
Phase 6 changes labels into summaries: cross-entropy and autoregressive decoding
replace Phase 5 BCE, label thresholds and statute-ID prediction.
"""
from __future__ import annotations
import contextlib, hashlib, json, math, os, random, re, shutil, tempfile, time, warnings, zipfile
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Optional
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from importlib import import_module

DATASET_REPO = 'Exploration-Lab/CS-781-Capstone'
DATASET_FILES = {'train':'Phase_6_Summ/Train/train.json','test':'Phase_6_Summ/Test/test.json'}
DEFAULT_REPOS = {
    'a1':'P-261110044/cs781-a1-checkpoints',
    'a2':'P-261110044/Phase1-A2',
    'a3':'P-261110044/cs781-phase1-architecture3',
    'a4':'P-261110044/cs781-a4-checkpoints',
}
A4_PATTERNS = [
 {'type':'local','value':256},{'type':'dilated','value':2},
 {'type':'local','value':512},{'type':'dilated','value':4},
 {'type':'local','value':512},{'type':'dilated','value':8},
 {'type':'local','value':1024},{'type':'global','value':None},
]
MODEL_DEFAULTS = {
 'a1':dict(context_length=2048,d_model=640,n_heads=10,n_layers=8,d_ff=2560,dropout=0.1),
 'a2':dict(source_length=1536,target_length=512,d_model=600,n_heads=15,n_encoder_layers=8,n_decoder_layers=8,d_ff=2048,dropout=0.1),
 'a3':dict(dim=600,num_layers=8,num_heads=10,num_kv_heads=5,ffn_dim=1600,window_size=512,chunk_size=256,dropout=0.1),
 'a4':dict(context_length=2048,dim=600,num_heads=10,ffn_dim=1600,attention_patterns=A4_PATTERNS,dropout=0.1),
}
PREFERENCE = {
 'a1':['checkpoint-6104/model.safetensors','final_model/model_state.pt','model_state.pt'],
 'a2':['cs781_phase1_a2_encoderdecoder/legal-gpt2-final/model_state.pt','legal-gpt2-final/model_state.pt','model_state.pt','final_model/model_state.pt'],
 'a3':['architecture-3-final/model_state.pt','model_state.pt','final_model/model_state.pt'],
 'a4':['final_model/model_state.pt','architecture-4-final/model_state.pt','model_state.pt'],
}

def set_seed(seed:int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def save_json(path, obj):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    try:
        with open(tmp,'w',encoding='utf-8') as f:
            json.dump(obj,f,ensure_ascii=False,indent=2,allow_nan=False)
        os.replace(tmp,path)
    finally:
        tmp.unlink(missing_ok=True)

def read_json(path):
    with open(path,encoding='utf-8') as f:return json.load(f)

def sha_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()

def normalize_rows(raw, split, return_audit=False):
    """Validate official IDs; skip unusable TRAIN supervision with a complete audit.

    A missing/blank document or reference summary cannot contribute a training
    objective, so it is excluded only from the public TRAIN split. Duplicate IDs,
    malformed JSON structures, and bad TEST documents remain hard errors.
    """
    if isinstance(raw, dict):
        if 'data' in raw and isinstance(raw['data'], list):
            raw = raw['data']
        elif all(isinstance(v, dict) for v in raw.values()):
            raw = [dict(v, id=k) if 'id' not in v else v for k, v in raw.items()]
        else:
            raise ValueError(f'{split}: unsupported JSON object; expected list of records')
    if not isinstance(raw, list):
        raise ValueError(f'{split}: expected list of records')

    results, ids, excluded = [], set(), []
    for i, r in enumerate(raw):
        if not isinstance(r, dict):
            raise ValueError(f'{split}[{i}]: expected JSON object')
        if 'id' not in r:
            raise ValueError(f'{split}[{i}]: missing id')
        ident = r['id']
        if not isinstance(ident, (str, int)) or isinstance(ident, bool) or not str(ident):
            raise ValueError(f'{split}[{i}]: invalid ID {ident!r}')
        ident = str(ident)
        if ident in ids:
            raise ValueError(f'{split}: duplicate document id {ident!r}')
        ids.add(ident)

        doc = r.get('document')
        invalid_document = not isinstance(doc, str) or not doc.strip()
        if split == 'test':
            if invalid_document:
                raise ValueError(f'test id={ident}: document must be a nonempty string')
            if 'summary' in r and r['summary'] not in (None, ''):
                raise ValueError(f'test id={ident}: unexpected reference summary; wrong split?')
            results.append({'id': ident, 'document': doc})
            continue

        summary = r.get('summary')
        invalid_summary = not isinstance(summary, str) or not summary.strip()
        if invalid_document or invalid_summary:
            # Never silently fabricate gold summaries or train on empty targets.
            reasons = []
            if invalid_document:
                reasons.append('empty_or_nontext_document')
            if invalid_summary:
                reasons.append('empty_or_nontext_summary')
            excluded.append({'id': ident, 'row_index': i, 'reasons': reasons})
            continue
        results.append({'id': ident, 'document': doc, 'summary': summary})

    if not results:
        raise ValueError(f'{split}: no usable records (raw={len(raw)}, excluded={len(excluded)})')
    audit = {'split': split, 'raw_records': len(raw), 'usable_records': len(results),
             'excluded_count': len(excluded), 'excluded_records': excluded}
    return (results, audit) if return_audit else results

def locate_dataset_file(split, data_dir='', token=None):
    rel=DATASET_FILES[split]
    if data_dir:
        root=Path(data_dir)
        options=[root/rel,root/Path(rel).name,root/('Train' if split=='train' else 'Test')/Path(rel).name]
        for p in options:
            if p.is_file():return p.resolve()
        raise FileNotFoundError(f'Cannot find {rel} under --data-dir {data_dir}. Looked in: {options}')
    # Search Kaggle mounted Input first. Avoid accidental train/test basename collisions.
    kaggle=Path('/kaggle/input')
    if kaggle.is_dir():
        found=sorted(kaggle.rglob(rel))
        if len(found)==1:return found[0]
        if len(found)>1:raise RuntimeError(f'Multiple copies of {rel} found in Kaggle inputs; choose --data-dir explicitly: {found[:5]}')
    try:
        from huggingface_hub import hf_hub_download
        return Path(hf_hub_download(repo_id=DATASET_REPO,filename=rel,repo_type='dataset',token=token))
    except Exception as e:
        raise FileNotFoundError(f'Cannot obtain official {rel}; mount the dataset in Kaggle, set --data-dir, or enable Internet. Cause: {e}') from e

def read_data(split,data_dir='',token=None):
    p=locate_dataset_file(split,data_dir,token)
    records,quality=normalize_rows(read_json(p),split,return_audit=True)
    if quality['excluded_count']:
        print(f"TRAIN DATA: excluded {quality['excluded_count']} unusable records; "
              f"using {quality['usable_records']}/{quality['raw_records']}. "
              'All excluded IDs/reasons are saved to runs/<method>/data_audit.json.',flush=True)
    return records,dict(file=str(p),sha256=sha_file(p),data_validation=quality)

def read_examples(split, data_dir='', token=None):
    # Phase-5-style public entry point; read_data also returns dataset SHA for auditing.
    examples, _ = read_data(split, data_dir, token)
    return examples


def partition_train(records, seed=44, fraction=.1):
    """Split by entire original document. Remove exact duplicate text; keep originals in audit."""
    if not 0<fraction<.5:raise ValueError('dev fraction must be in (0,0.5)')
    uniq=[]; duplicates=[]; seen={}
    for row in records:
        digest=hashlib.sha256(row['document'].encode('utf-8')).hexdigest()
        if digest in seen:
            prev=seen[digest]
            if prev['summary']!=row['summary']:
                # Conflicting summaries for the same input: leave out the duplicate, audit it.
                duplicates.append({'id':row['id'],'canonical_id':prev['id'],'reason':'same text, differing summary'})
            else: duplicates.append({'id':row['id'],'canonical_id':prev['id'],'reason':'identical text and summary'})
            continue
        seen[digest]=row;uniq.append(row)
    if len(uniq)<3:raise ValueError('At least three distinct training documents are required for a holdout')
    order=list(range(len(uniq)));random.Random(seed).shuffle(order)
    size=max(1,min(len(uniq)-1,round(len(uniq)*fraction)))
    dev_idx=set(order[:size]); train=[r for i,r in enumerate(uniq) if i not in dev_idx]; dev=[r for i,r in enumerate(uniq) if i in dev_idx]
    assert {r['id'] for r in train}.isdisjoint({r['id'] for r in dev})
    assert {hashlib.sha256(r['document'].encode()).hexdigest() for r in train}.isdisjoint({hashlib.sha256(r['document'].encode()).hexdigest() for r in dev})
    return train,dev,dict(excluded_duplicate_train_records=duplicates,n_train=len(train),n_dev=len(dev),seed=seed,dev_fraction=fraction)

def normalize_repo(name,arch):
    return name or os.environ.get('PHASE1_REPO_'+arch.upper()) or DEFAULT_REPOS[arch]

def get_hf_token(required=False):
    for k in ('HF_TOKEN','HUGGING_FACE_HUB_TOKEN','HF'):
        if os.environ.get(k):
            # Also authenticate nested HF downloads (e.g. the BERTScore scorer).
            os.environ.setdefault('HF_TOKEN', os.environ[k])
            return os.environ[k]
    try:
        from kaggle_secrets import UserSecretsClient
        sec=UserSecretsClient()
        for k in ('HF','HF_TOKEN','HUGGING_FACE_HUB_TOKEN'):
            try:
                t=sec.get_secret(k)
                if t:
                    # Propagate Kaggle's HF secret to libraries that read HF_TOKEN.
                    os.environ['HF_TOKEN'] = t
                    return t
            except (KeyError,RuntimeError,ValueError):pass
    except (ImportError,ModuleNotFoundError):pass
    if required:raise RuntimeError('HF token unavailable. Add Kaggle Secret named HF and enable its access.')
    return None

def select_weight_name(files,arch,explicit=''):
    fs=set(files)
    if explicit:
        if explicit not in fs:raise FileNotFoundError(f'Chosen --phase1-weight {explicit} missing; examples: {sorted(fs)[:12]}')
        return explicit
    for choice in PREFERENCE[arch]:
        if choice in fs:return choice
    # Phase-1 artifacts can live inside a training-output prefix in the HF repository.
    for suffix in PREFERENCE[arch]:
        suffix_matches=[f for f in files if f.endswith('/'+suffix)]
        if len(suffix_matches)==1:return suffix_matches[0]
        if len(suffix_matches)>1:
            raise RuntimeError(f'Ambiguous Phase1 checkpoint suffix {suffix}: {suffix_matches}. Specify --phase1-weight.')
    candidates=[s for s in fs if s.endswith(('.safetensors','model_state.pt','pytorch_model.bin')) and
                not re.search(r'optimizer|scheduler|adapter|phase[234567]|best\.pt',s,re.I)]
    if not candidates:raise FileNotFoundError('No Phase-1 weights found; specify explicit --phase1-weight.')
    steps=[c for c in candidates if re.search(r'checkpoint-\d+/model\.safetensors$',c)]
    if steps:
        return max(steps,key=lambda v:int(re.search(r'checkpoint-(\d+)',v).group(1)))
    if len(candidates)>1:
        raise RuntimeError(f'Ambiguous Phase-1 weights {candidates[:12]}; use --phase1-weight.')
    return candidates[0]

def prepare_phase1(arch,repo='',local_dir='',weight='',token=None,cache_dir=''):
    """Find matched Phase-1 checkpoint and tokenizer. Never substitute another pretrained model."""
    local_dir=local_dir.strip();repo=normalize_repo(repo,arch)
    cache=Path(cache_dir or ('./phase1_cache'))/arch
    cache.mkdir(parents=True,exist_ok=True)
    if local_dir:
        root=Path(local_dir)
        if not root.is_dir():raise FileNotFoundError(f'Phase1 local directory missing: {root}')
        files=[p.relative_to(root).as_posix() for p in root.rglob('*') if p.is_file()]
        def resolve(rel):return root/rel
        source=str(root.resolve())
    else:
        try:
            from huggingface_hub import HfApi,hf_hub_download
            api=HfApi(token=token)
            files=api.list_repo_files(repo_id=repo,repo_type='model')
        except Exception as e:raise RuntimeError(f'Cannot access original Phase1 repo {repo!r}. Configure HF token/Internet or --phase1-dir. Cause: {e}') from e
        def resolve(rel):
            return Path(hf_hub_download(repo_id=repo,filename=rel,repo_type='model',token=token,local_dir=str(cache)))
        source=repo
    rel=select_weight_name(files,arch,weight)
    wpath=resolve(rel)
    folders=[]
    folder=rel.rsplit('/',1)[0]+'/' if '/' in rel else ''
    folders.extend([folder, 'final_model/','architecture-3-final/','architecture-4-final/','legal-gpt2-final/',''])
    files_set=set(files)
    tok_rel=next((f+'tokenizer.json' for f in dict.fromkeys(folders) if f+'tokenizer.json' in files_set),None)
    if not tok_rel:
        matches=[f for f in files if f.endswith('/tokenizer.json') or f=='tokenizer.json']
        if len(matches)==1:tok_rel=matches[0]
        elif matches:raise RuntimeError(f'Multiple unmatched tokenizer.json in {source}: {matches}. Supply a Phase1 directory with an unambiguous tokenizer.')
        else:raise FileNotFoundError(f'No tokenizer.json in Phase 1 source {source}; do not substitute a tokenizer.')
    tokenizer_path=resolve(tok_rel);tokdir=tokenizer_path.parent
    for fname in ('tokenizer_config.json','special_tokens_map.json','added_tokens.json'):
        path=tok_rel.rsplit('/',1)[0]+'/' +fname if '/' in tok_rel else fname
        if path in files_set:resolve(path)
    cfg_rel=next((f+'architecture_config.json' for f in dict.fromkeys(folders) if f+'architecture_config.json' in files_set),None)
    config=MODEL_DEFAULTS[arch].copy();config_source='Phase-1 notebook defaults'
    if cfg_rel:
        cfg=read_json(resolve(cfg_rel));aliases={
         'n_embd':'dim' if arch in ('a3','a4') else 'd_model',
         'n_layer':'num_layers' if arch=='a3' else 'n_layers' if arch=='a1' else 'n_layer',
         'n_head':'num_heads' if arch=='a3' else 'n_heads',
         'ffn_dim':'ffn_dim',
        }
        for k,v in cfg.items():
            target=aliases.get(k,k)
            if target in config:config[target]=v
        config_source=cfg_rel
    state=get_model_state(wpath)
    inferred,vocab=infer_phase1_config(arch,state,config)
    del state
    return dict(arch=arch,source=source,weights=str(wpath),weight_relative=rel,tokenizer_dir=str(tokdir),
                tokenizer_sha=sha_file(tokenizer_path),config=inferred,checkpoint_vocab=vocab,
                config_source=config_source+' + checkpoint tensor shapes',weight_sha=sha_file(wpath))

def load_tokenizer(bundle):
    from transformers import PreTrainedTokenizerFast
    path=Path(bundle['tokenizer_dir'])
    if not (path/'tokenizer.json').is_file():raise FileNotFoundError(f'Tokenizer missing: {path}')
    # The exact Phase1 notebook registered these four existing BPE special tokens.
    tok=PreTrainedTokenizerFast(tokenizer_file=str(path/'tokenizer.json'),pad_token='<pad>',unk_token='<unk>',bos_token='<s>',eos_token='</s>',
                                     clean_up_tokenization_spaces=False)
    if not all(isinstance(getattr(tok,k+'_token_id'),int) for k in ('bos','eos','pad','unk')):
        raise ValueError('Phase-1 tokenizer missing special IDs')
    if len({tok.bos_token_id,tok.eos_token_id,tok.pad_token_id,tok.unk_token_id})!=4:
        raise ValueError('Phase1 special tokens are not four distinct IDs; incompatible tokenizer.')
    return tok

def create_model(arch,vocab_size,config=None,student_layers=0):
    cfg=(config or MODEL_DEFAULTS[arch]).copy()
    if student_layers:
        if arch!='a3':raise ValueError('student_layers supported only for A3')
        if not 1<=student_layers<cfg['num_layers']:raise ValueError('student_layers must be fewer than teacher layers')
        cfg['num_layers']=student_layers
    names={'a1':('a1','DecoderTransformerLM'),
           'a2':('a2','EncoderDecoderTransformer'),
           'a3':('a3','ModernLocalTransformerLM'),
           'a4':('a4','MultiScaleDilatedLM')}
    if arch not in names:raise ValueError(f'Unknown architecture {arch!r}')
    module,classname=names[arch]
    model_class=getattr(import_module('phase6_backbones.'+module),classname)
    return model_class(vocab_size=vocab_size,**cfg)

def get_model_state(path):
    path=Path(path)
    if path.suffix=='.safetensors':
        from safetensors.torch import load_file
        state=load_file(str(path),device='cpu')
    else:
        state=torch.load(path,map_location='cpu',weights_only=True)
        if isinstance(state,dict):
            for key in ('model','model_state','state_dict','model_state_dict'):
                if key in state and isinstance(state[key],dict):state=state[key];break
    if not isinstance(state,dict) or not state:raise ValueError(f'{path}: checkpoint does not contain model tensor dictionary')
    for prefix in ('module.','_orig_mod.'):
        if state and all(k.startswith(prefix) for k in state):
            state={k[len(prefix):]:v for k,v in state.items()}
    return state

def infer_phase1_config(arch,state,prior):
    """Infer tensor-shaped architecture fields from ACTUAL checkpoint, not stale notebook defaults.
    Head counts and sparse-attention patterns cannot be inferred from weights; use saved config
    when available and otherwise the original notebook's architectural choice.
    """
    cfg=prior.copy()
    if 'token_embedding.weight' not in state:
        raise RuntimeError('Phase1 checkpoint missing token_embedding.weight')
    vocab,width=state['token_embedding.weight'].shape
    def count(prefix):
        ids={int(m.group(1)) for k in state if (m:=re.match('^'+re.escape(prefix)+r'\.(\d+)\.',k))}
        if not ids or ids!=set(range(max(ids)+1)):
            raise RuntimeError(f'Checkpoint layer indices for {prefix} are missing/non-contiguous: {sorted(ids)}')
        return len(ids)
    def require_shape(key):
        if key not in state:raise RuntimeError(f'Checkpoint missing architecture-defining weight {key}')
        return state[key].shape
    if arch=='a1':
        cfg.update(d_model=width,n_layers=count('blocks'),context_length=require_shape('position_embedding.weight')[0],
                   d_ff=require_shape('blocks.0.feed_forward.0.weight')[0])
    elif arch=='a2':
        cfg.update(d_model=width,n_encoder_layers=count('encoder_blocks'),n_decoder_layers=count('decoder_blocks'),
                   source_length=require_shape('source_position.weight')[0],target_length=require_shape('target_position.weight')[0],
                   d_ff=require_shape('encoder_blocks.0.feed_forward.0.weight')[0])
    elif arch=='a3':
        cfg.update(dim=width,num_layers=count('blocks'),ffn_dim=require_shape('blocks.0.ffn.gate_proj.weight')[0])
    elif arch=='a4':
        cfg.update(dim=width,context_length=require_shape('position_embedding.weight')[0],
                   ffn_dim=require_shape('blocks.0.ffn.0.weight')[0])
        layers=count('blocks')
        if len(cfg['attention_patterns'])!=layers:
            raise RuntimeError(f'A4 has {layers} layers but only {len(cfg["attention_patterns"])} saved attention patterns; provide correct Phase1 architecture_config.json')
    else:raise ValueError(arch)
    heads=cfg.get('n_heads',cfg.get('num_heads'))
    if not isinstance(heads,int) or heads<1 or width%heads:
        raise RuntimeError(f'Invalid number of Phase1 attention heads ({heads}) for checkpoint width {width}. Restore original Phase1 config.')
    if arch=='a3':
        if cfg['num_heads']%cfg['num_kv_heads'] or (width//heads)%2:
            raise RuntimeError('A3 GQA/RoPE architecture config disagrees with checkpoint shape')
    return cfg,int(vocab)

def load_phase1_weights(model,path,student_layers=0):
    state=get_model_state(path)
    if student_layers:
        original=set(state); target=model.state_dict()
        extra=[k for k in original if k not in target]
        if any(not re.match(r'^blocks\.\d+\.',k) for k in extra):
            raise RuntimeError(f'Unexpected non-block keys from teacher checkpoint: {extra[:12]}')
        state={k:v for k,v in state.items() if k in target}
    expected=model.state_dict()
    if 'lm_head.weight' in state and 'lm_head.weight' not in expected and 'output_layer.weight' in expected:
        if 'output_layer.weight' not in state:state['output_layer.weight']=state['lm_head.weight']
        del state['lm_head.weight']
    for key,other in (('lm_head.weight','token_embedding.weight'),('token_embedding.weight','lm_head.weight')):
        if key in expected and key not in state and other in state and expected[key].shape==state[other].shape:
            state[key]=state[other]
    missing=sorted(set(expected)-set(state));extra=sorted(set(state)-set(expected))
    wrong=[(k,tuple(state[k].shape),tuple(expected[k].shape)) for k in set(expected)&set(state) if state[k].shape!=expected[k].shape]
    if missing or extra or wrong:
        raise RuntimeError(f'Phase1 checkpoint INCOMPATIBLE; missing={missing[:8]}, extra={extra[:8]}, shapes={wrong[:8]}. Check architecture config and tokenizer. No partial loading.')
    model.load_state_dict(state,strict=True)
    del state

def replace_modules(model,matcher,make_wrapper):
    count=0
    for name,child in list(model.named_children()):
        if isinstance(child,nn.Linear) and matcher(name,child):
            setattr(model,name,make_wrapper(child));count+=1
        elif not isinstance(child,(LoRALinear,LoRMALinear)):
            count+=replace_modules(child,matcher,make_wrapper)
    return count

class LoRALinear(nn.Module):
    def __init__(self,base,rank=8,alpha=16):
        super().__init__();self.base=base;self.rank=rank;self.scaling=alpha/rank
        self.A=nn.Parameter(torch.empty(rank,base.in_features));self.B=nn.Parameter(torch.zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))
        for p in self.base.parameters():p.requires_grad=False
    def forward(self,x):return self.base(x)+F.linear(F.linear(x,self.A),self.B)*self.scaling

class LoRMALinear(nn.Module):
    """LoRMA+ identity-inflated multiplicative update (I + alpha/r BA) W0, not additive LoRA.
    Base linear bias is unchanged. No unstable full-rank matrix materialization needed.
    """
    def __init__(self,base,rank=8,alpha=16):
        super().__init__();self.base=base;self.rank=rank;self.scaling=alpha/rank
        self.A=nn.Parameter(torch.empty(rank,base.out_features));self.B=nn.Parameter(torch.zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))
        for p in self.base.parameters():p.requires_grad=False
    def forward(self,x):
        linear_no_bias=F.linear(x,self.base.weight,None)
        return self.base(x)+F.linear(F.linear(linear_no_bias,self.A),self.B)*self.scaling

class ResidualAdapterBlock(nn.Module):
    """Wrap original block without overwriting or text-patching original forward method.
    Supports both tensor and tuple (hidden, cache) outputs from A1/A3/A4 blocks.
    """
    def __init__(self,inner,dim,bottleneck=32):
        super().__init__();self.inner=inner
        self.down=nn.Linear(dim,bottleneck);self.up=nn.Linear(bottleneck,dim)
        nn.init.zeros_(self.up.weight);nn.init.zeros_(self.up.bias)
        for p in self.inner.parameters():p.requires_grad=False
    def forward(self,*args,**kwargs):
        out=self.inner(*args,**kwargs)
        def augment(x):return x+self.up(F.gelu(self.down(x)))
        if isinstance(out,tuple):return (augment(out[0]),)+out[1:]
        return augment(out)

def adapt_model(model,arch,method,rank=8,alpha=16,adapter_size=32):
    if method not in ('full','partial','lora','lorma','adapters','kd'):raise ValueError(f'Unknown method {method}')
    for p in model.parameters():p.requires_grad=(method=='full')
    if method=='full':pass
    elif method=='partial':
        if arch!='a2':raise ValueError('Partial fine-tuning experiment must use A2')
        groups=[model.encoder_blocks[-1],model.decoder_blocks[-1],model.final_norm,model.output_layer]
        for g in groups:
            for p in g.parameters():p.requires_grad=True
    elif method in ('lora','lorma'):
        if method=='lora' and arch!='a3':raise ValueError('LoRA experiment expects A3')
        if method=='lorma' and arch!='a4':raise ValueError('LoRMA experiment expects A4')
        wrapper=LoRALinear if method=='lora' else LoRMALinear
        # Target real linear projections, exclude vocab head and output embeddings.
        for block in model.blocks:
            n=replace_modules(block,lambda name,c: name in ('q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'),lambda old:wrapper(old,rank,alpha))
            if not n:raise RuntimeError('No attention/FFN projection layers found to adapt')
    elif method=='adapters':
        if arch!='a1':raise ValueError('Adapters experiment expects A1')
        for i,block in enumerate(model.blocks):model.blocks[i]=ResidualAdapterBlock(block,model.token_embedding.embedding_dim,adapter_size)
    elif method=='kd':
        if arch!='a3':raise ValueError('KD student uses A3')
        for p in model.parameters():p.requires_grad=True
    trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
    if not trainable:raise RuntimeError(f'{method}: zero trainable parameters')
    return {'total_params':sum(p.numel() for p in model.parameters()),'trainable_params':trainable,'trainable_fraction':trainable/sum(p.numel() for p in model.parameters())}

PROMPT_START='Summarize the following legal document, preserving facts, arguments, reasoning and the ruling.\nDOCUMENT:\n'
PROMPT_END='\nLEGAL SUMMARY:\n'

def doc_to_tokens(tokenizer,doc,max_doc_tokens=448,max_chars=120000,strategy='head_middle_tail'):
    if strategy not in ('head_tail','head_middle_tail'):raise ValueError('Unknown document selection strategy')
    if len(doc)>max_chars:
        # Use both ends instead of dropping conclusion/holding at end.
        half=max_chars//2;doc=doc[:half]+'\n... [document gap] ...\n'+doc[-half:]
    tokens=tokenizer.encode(doc,add_special_tokens=False)
    if len(tokens)<=max_doc_tokens:return tokens
    if strategy=='head_tail':
        first=round(max_doc_tokens*.56);return tokens[:first]+tokens[-(max_doc_tokens-first):]
    a=max_doc_tokens//3;b=max_doc_tokens//3;c=max_doc_tokens-a-b
    mid=len(tokens)//2;start=max(0,mid-b//2)
    return tokens[:a]+tokens[start:start+b]+tokens[-c:]

def make_prompt(tokenizer,doc,max_source=512,strategy='head_middle_tail'):
    begin=tokenizer.encode(PROMPT_START,add_special_tokens=False)
    end=tokenizer.encode(PROMPT_END,add_special_tokens=False)
    if len(begin)+len(end)+16>=max_source:raise ValueError('max_source too small for prompt')
    docids=doc_to_tokens(tokenizer,doc,max_doc_tokens=max_source-len(begin)-len(end),strategy=strategy)
    ret=begin+docids+end
    if not ret:raise ValueError('Empty prompt')
    assert len(ret)<=max_source
    return ret

def batch_tokens(tokenizer,record,arch,max_source=512,max_target=128,strategy='head_middle_tail'):
    source=make_prompt(tokenizer,record['document'],max_source,strategy)
    target=tokenizer.encode(record['summary'],add_special_tokens=False)[:max_target-1]+[tokenizer.eos_token_id]
    if not target:raise ValueError(f'Empty target for {record["id"]}')
    if arch=='a2':
        if len(source)>1536 or len(target)>512:raise ValueError('A2 source/target exceeds Phase-1 positions')
        return {'input_ids':torch.tensor(source,dtype=torch.long)[None,:],
                'targets':torch.tensor(target,dtype=torch.long)[None,:]}
    sequence=source+target
    if len(sequence)>2048:raise ValueError('Causal context exceeds Phase1 context')
    labels=[-100]*len(source)+target
    return {'input_ids':torch.tensor(sequence,dtype=torch.long)[None,:],
            'labels':torch.tensor(labels,dtype=torch.long)[None,:]}

def target_loss(model,arch,batch,tokenizer,return_logits=False):
    if arch=='a2':
        src=batch['input_ids']; tgt=batch['targets']
        decin=torch.cat([torch.full((tgt.shape[0],1),tokenizer.bos_token_id,dtype=torch.long,device=tgt.device),tgt[:,:-1]],dim=1)
        enc=model.encode(src);logits=model.decode(decin,enc)
        loss=F.cross_entropy(logits.reshape(-1,logits.size(-1)).float(),tgt.reshape(-1))
        mask=torch.ones(tgt.shape,device=tgt.device,dtype=torch.bool)
        return (loss,logits,mask) if return_logits else loss
    out=model(batch['input_ids'])
    logits=out.logits[:,:-1,:]; y=batch['labels'][:,1:]
    mask=y!=-100
    if not mask.any():raise ValueError('No unmasked summary tokens: source/target alignment error')
    loss=F.cross_entropy(logits.reshape(-1,logits.size(-1)).float(),y.reshape(-1),ignore_index=-100)
    return (loss,logits,mask) if return_logits else loss

def _blocked_ngram_tokens(output, n):
    """Tokens that would repeat a generated n-gram; never inspect the input document."""
    if n == 0:
        return set()
    if n < 2:
        raise ValueError('no_repeat_ngram must be 0 (off) or >=2')
    if len(output) < n - 1:
        return set()
    prefix = tuple(output[-(n - 1):])
    return {output[i + n - 1] for i in range(len(output) - n + 1)
            if tuple(output[i:i + n - 1]) == prefix}


def _greedy_token(logits, eos_ids=(), force_continue=False, output=(), no_repeat_ngram_size=0):
    if not torch.isfinite(logits).all():
        raise FloatingPointError('Non-finite generation logits detected')
    if force_continue or no_repeat_ngram_size:
        logits = logits.clone()  # Never mutate a model's cached logits.
    if force_continue:
        for ix in eos_ids:
            if ix is not None and 0 <= int(ix) < logits.shape[-1]:
                logits[..., int(ix)] = -float('inf')
    for ix in _blocked_ngram_tokens(output, no_repeat_ngram_size):
        if 0 <= int(ix) < logits.shape[-1]:
            logits[..., int(ix)] = -float('inf')
    if not torch.isfinite(logits).any():
        raise FloatingPointError('No valid next token after decoding constraints')
    return int(logits.argmax(dim=-1).item())

@torch.inference_mode()
def generate(model,arch,tokenizer,doc,max_source=512,max_new=128,strategy='head_middle_tail',device=None,
             no_repeat_ngram_size=0):
    if no_repeat_ngram_size != 0 and not 2 <= no_repeat_ngram_size <= max_new:
        raise ValueError('no_repeat_ngram_size must be 0 or between 2 and max_new')
    device=device or next(model.parameters()).device
    prompt=make_prompt(tokenizer,doc,max_source,strategy)
    x=torch.tensor([prompt],device=device,dtype=torch.long)
    model.eval();output=[]
    eos_ids=(tokenizer.eos_token_id,tokenizer.pad_token_id)
    min_new=min(12,max_new)
    if arch=='a2':
        enc=model.encode(x);dec=torch.tensor([[tokenizer.bos_token_id]],device=device)
        for _ in range(min(max_new,model.target_length)):
            logits=model.decode(dec,enc)
            token=_greedy_token(logits[:,-1,:],eos_ids,len(output)<min_new,output,no_repeat_ngram_size);
            if token in (tokenizer.eos_token_id,tokenizer.pad_token_id):break
            output.append(token)
            if dec.shape[1]>=model.target_length:break
            dec=torch.cat((dec,torch.tensor([[token]],device=device)),dim=1)
    elif arch in ('a3','a4') and hasattr(model,'prefill'):
        if arch=='a4' and len(prompt)+max_new>model.context_length:
            raise ValueError('A4 generation would exceed Phase1 context')
        logits,cache=model.prefill(x)
        for step in range(max_new):
            tok=_greedy_token(logits[:,-1,:],eos_ids,len(output)<min_new,output,no_repeat_ngram_size);
            if tok in (tokenizer.eos_token_id,tokenizer.pad_token_id):break
            output.append(tok)
            if step==max_new-1:break
            # Original A3 and A4 each provide Phase1 KV-cache decode functions.
            logits,cache=model.decode_step(torch.tensor([[tok]],device=device),cache,position_offset=len(prompt)+step)
            if logits.ndim==2:logits=logits[:,None,:]
    else:
        for _ in range(max_new):
            if x.shape[1]>=model.context_length:break
            logits=model(x).logits
            tok=_greedy_token(logits[:,-1,:],eos_ids,len(output)<min_new,output,no_repeat_ngram_size);
            if tok in (tokenizer.eos_token_id,tokenizer.pad_token_id):break
            output.append(tok);x=torch.cat((x,torch.tensor([[tok]],device=device)),dim=1)
    # Preserve BPE punctuation spacing (no WordPiece cleanup).
    text=tokenizer.decode(output,skip_special_tokens=True,clean_up_tokenization_spaces=False).strip()
    # strip generative artifacts while retaining content; no extraction from input.
    text=re.sub(r'\s+',' ',text).strip()
    if not text:
        # A blank output is never silently submitted: force user to train/fix generation.
        raise ValueError('Model produced an empty summary. Inspect checkpoint/decoding rather than inventing a fallback.')
    return text

def repetition_summary(summaries, n=4, high_threshold=0.25):
    """Non-scoring quality diagnostic; flags repeated word n-grams in generated text."""
    if not summaries:
        raise ValueError('Cannot audit an empty list of summaries')
    rates = []
    for text in summaries:
        words = re.findall(r"\b\w+(?:['’-]\w+)*\b", text.lower())
        ngrams = [tuple(words[i:i+n]) for i in range(max(0, len(words)-n+1))]
        rates.append((len(ngrams)-len(set(ngrams)))/len(ngrams) if ngrams else 0.)
    return {'ngram_size':n,'average_repeated_fraction':round(sum(rates)/len(rates),4),
            'high_repetition_count':sum(x>high_threshold for x in rates),
            'high_threshold':high_threshold,'total':len(summaries)}

def tokenize_metric(s):return re.findall(r"\b\w+(?:['’-]\w+)*\b",s.lower())

def rouge1(reference,hypothesis):
    a=Counter(tokenize_metric(reference));b=Counter(tokenize_metric(hypothesis));hit=sum((a&b).values())
    den=sum(a.values())+sum(b.values());return (2*hit/den) if den else 0.

def rouge_l(reference,hypothesis):
    a=tokenize_metric(reference);b=tokenize_metric(hypothesis)
    if not a or not b:return 0.
    if len(b)>len(a):a,b=b,a
    dp=[0]*(len(b)+1)
    for ta in a:
        prev=0
        for j,tb in enumerate(b,1):
            cur=dp[j];dp[j]=(prev+1) if ta==tb else max(dp[j],dp[j-1]);prev=cur
    return 2*dp[-1]/(len(a)+len(b))

def evaluate_pairs(reference,prediction):
    if len(reference)!=len(prediction) or not reference:raise ValueError('Evaluation predictions/references size mismatch')
    return {'rouge1_f1':round(sum(rouge1(r,p) for r,p in zip(reference,prediction))/len(reference),6),
            'rougeL_f1':round(sum(rouge_l(r,p) for r,p in zip(reference,prediction))/len(reference),6),
            'n':len(reference)}

def optional_bertscore(refs,preds,enabled=False):
    if not enabled:return None
    try:
        from bert_score import score
    except ImportError as e:raise RuntimeError('Optional BERTScore requires pip install bert-score, and a scorer model download; it is never used for generation.') from e
    p,r,f=score(preds,refs,lang='en',verbose=False,device='cuda' if torch.cuda.is_available() else 'cpu')
    return float(f.mean().item())

def validate_submission(preds,test_records):
    if not isinstance(preds,list):raise ValueError('Phase6 list format required')
    required=[str(r['id']) for r in test_records]
    pred_ids=[]
    for i,row in enumerate(preds):
        if not isinstance(row,dict) or set(row)!={'id','summary'}:raise ValueError(f'prediction[{i}]: required keys exactly id, summary')
        if not isinstance(row['id'],str):raise ValueError(f'prediction[{i}]: ID must be string to preserve official format')
        if not isinstance(row['summary'],str) or not row['summary'].strip():raise ValueError(f'prediction[{i}]: blank/nonstring summary')
        pred_ids.append(row['id'])
    if len(set(pred_ids))!=len(pred_ids):raise ValueError('Duplicate prediction IDs')
    if set(pred_ids)!=set(required):
        raise ValueError(f'ID mismatch: missing {list(set(required)-set(pred_ids))[:8]}, unknown {list(set(pred_ids)-set(required))[:8]}')
    if pred_ids!=required:raise ValueError('Prediction record order not same as official test file; preserve exact order')
    return {'n_examples':len(preds),'all_ids_exact':True,'all_summaries_nonempty':True,'order_preserved':True}

def package_submission(preds,test_records,outdir):
    outdir=Path(outdir);outdir.mkdir(parents=True,exist_ok=True)
    report=validate_submission(preds,test_records)
    prediction=outdir/'prediction.json';archive=outdir/'submission.zip'
    save_json(prediction,preds)
    tmp=outdir/'submission.zip.tmp'
    try:
        with zipfile.ZipFile(tmp,'w',compression=zipfile.ZIP_DEFLATED) as z:z.write(prediction,arcname='prediction.json')
        with zipfile.ZipFile(tmp) as z:
            if z.namelist()!=['prediction.json']:raise RuntimeError('zip root layout invalid')
            payload=json.loads(z.read('prediction.json').decode('utf-8'))
            validate_submission(payload,test_records)
        os.replace(tmp,archive)
    finally:tmp.unlink(missing_ok=True)
    save_json(outdir/'submission_audit.json',{**report,'prediction_sha256':sha_file(prediction),'submission_sha256':sha_file(archive)})
    return prediction,archive

def torch_load(path):return torch.load(path,map_location='cpu',weights_only=True)

def atomic_torch_save(obj,path):
    p=Path(path);p.parent.mkdir(parents=True,exist_ok=True);tmp=p.with_name(p.name+'.tmp')
    try:
        torch.save(obj,tmp);os.replace(tmp,p)
    finally:tmp.unlink(missing_ok=True)

def make_training_checkpoint(model,optimizer,arch,method,settings,epoch,cursor,step,score=None):
    return {'version':1,'arch':arch,'method':method,'settings':settings,'epoch':epoch,'cursor':cursor,
            'global_step':step,'score':score,
            'model':{k:v.detach().cpu() for k,v in model.state_dict().items()},
            'optimizer':optimizer.state_dict() if optimizer is not None else None,
            'rng_torch':torch.get_rng_state(),
            'rng_cuda':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}

def restore_training_checkpoint(ckpt,model,opt,arch,method,settings):
    if ckpt.get('version')!=1 or ckpt.get('arch')!=arch or ckpt.get('method')!=method:
        raise RuntimeError('Resume checkpoint version/architecture/method mismatch')
    if ckpt['settings']!=settings:raise RuntimeError('Resume settings or tokenizer/dataset fingerprint changed; refusing unsafe resume')
    model.load_state_dict(ckpt['model'],strict=True)
    if opt is not None and ckpt.get('optimizer'):opt.load_state_dict(ckpt['optimizer'])
    if isinstance(ckpt.get('rng_torch'),torch.Tensor):torch.set_rng_state(ckpt['rng_torch'])
    if torch.cuda.is_available() and ckpt.get('rng_cuda'):torch.cuda.set_rng_state_all(ckpt['rng_cuda'])
    return int(ckpt['epoch']),int(ckpt['cursor']),int(ckpt['global_step'])


In [ ]:
%%writefile phase6_train.py
"""Train Phase 6 from original Phase 1 weights (same entry point as Phase 5).
  python phase6_train.py --arch a1 --method full --out runs/a1_full
  python phase6_train.py --arch a2 --method partial --out runs/a2_partial
  python phase6_train.py --arch a4 --method lorma --out runs/a4_lorma
  python phase6_distill.py --teacher runs/a3_teacher/best.pt --out runs/a3_kd

Phase 5 trained a multi-label classifier; Phase 6 trains next-token summarization.
"""
from __future__ import annotations
import argparse, contextlib, json, os, random, sys, time, traceback
from pathlib import Path
import torch
from torch.nn import functional as F
from phase6_core import *

def parse_args():
    p=argparse.ArgumentParser()
    p.add_argument('--arch',required=True,choices=list(DEFAULT_REPOS))
    p.add_argument('--method',required=True,choices=['full','partial','lora','lorma','adapters','kd'])
    p.add_argument('--out',required=True)
    p.add_argument('--data-dir',default='')
    p.add_argument('--phase1-dir',default='')
    p.add_argument('--source','--phase1-repo',dest='phase1_repo',default='',help='Phase 1 HF repo or original repo override')
    p.add_argument('--weight','--phase1-weight',dest='phase1_weight',default='',help='Exact Phase 1 checkpoint path within repo')
    p.add_argument('--teacher-checkpoint',default='',help='Required for kd; trained Phase6 A3 LoRA best.pt')
    p.add_argument('--teacher-repo',default='',help='Optional private HF repo containing runs/a3_lora/best.pt')
    p.add_argument('--epochs',type=int,default=3)
    p.add_argument('--batch-size',type=int,default=1)
    p.add_argument('--accum',type=int,default=4)
    p.add_argument('--max-source',type=int,default=448)
    p.add_argument('--max-target',type=int,default=112)
    p.add_argument('--decode-length',type=int,default=112)
    p.add_argument('--dev-eval-max',type=int,default=64)
    p.add_argument('--dev-fraction',type=float,default=.08)
    p.add_argument('--strategy',choices=['head_middle_tail','head_tail'],default='head_middle_tail')
    p.add_argument('--seed',type=int,default=44)
    p.add_argument('--lr',type=float,default=None)
    p.add_argument('--weight-decay',type=float,default=.01)
    p.add_argument('--rank',type=int,default=8)
    p.add_argument('--alpha',type=int,default=16)
    p.add_argument('--adapter-size',type=int,default=32)
    p.add_argument('--student-layers',type=int,default=4)
    p.add_argument('--kd-temperature',type=float,default=2.0)
    p.add_argument('--kd-soft-weight',type=float,default=.5)
    p.add_argument('--save-every',type=int,default=300)
    p.add_argument('--bertscore',action='store_true',help='Optional local scorer; requires third-party model solely for evaluation')
    p.add_argument('--cpu-test',action='store_true',help='Only for tiny self-tests; not actual competition training')
    p.add_argument('--no-resume',action='store_true')
    return p.parse_args()

def validate_args(a):
    known={'full':'a1','partial':'a2','lora':'a3','lorma':'a4','adapters':'a1','kd':'a3'}
    if known[a.method]!=a.arch:raise ValueError(f'Method {a.method} must use Phase1 {known[a.method]} architecture')
    if a.batch_size!=1:raise ValueError('Batch size 1 required. Original Phase1 architectures do not mask padding.')
    if a.accum<1 or a.epochs<1 or a.dev_eval_max<1:raise ValueError('accum, epochs, dev eval max must be >=1')
    if not (48<=a.max_source<=1400 and 8<=a.max_target<=512):raise ValueError('Invalid token length; max source 48..1400, target 8..512')
    if a.arch=='a2' and a.max_source>1536:raise ValueError('A2 source > original Phase1 source positions')
    if a.arch!='a2' and a.max_source+a.max_target>2048:raise ValueError('Causal context >2048')
    if not 8<=a.decode_length<=256:raise ValueError('decode-length outside 8..256')
    if a.kd_temperature<=0 or not (0<=a.kd_soft_weight<=1):raise ValueError('Invalid KD configuration')
    if a.save_every<1:raise ValueError('save-every must be >=1')

def store_event(path,record):
    with open(path,'a',encoding='utf-8') as f:f.write(json.dumps(record,ensure_ascii=False,allow_nan=False)+'\n')

def amp_context(device):
    return torch.autocast(device_type='cuda',dtype=torch.float16) if device.type=='cuda' else contextlib.nullcontext()

def load_teacher(args,token,backbone_bundle,tokenizer,device):
    candidate=args.teacher_checkpoint
    if not candidate and args.teacher_repo:
        from huggingface_hub import hf_hub_download,HfApi
        files=HfApi(token=token).list_repo_files(args.teacher_repo,repo_type='model')
        options=[f for f in files if f=='runs/a3_lora/best.pt'] or [f for f in files if f=='best.pt']
        if not options:raise FileNotFoundError(f'Teacher repo {args.teacher_repo} lacks trained Phase6 A3 LoRA best.pt; cannot distil from Phase1 alone')
        if len(options)>1:raise RuntimeError(f'Ambiguous teacher candidates {options}; specify --teacher-checkpoint')
        candidate=hf_hub_download(repo_id=args.teacher_repo,filename=options[0],token=token)
    if not candidate or not Path(candidate).is_file():
        raise FileNotFoundError('KD requires Phase6-trained A3 LoRA teacher. Provide --teacher-checkpoint or --teacher-repo. Teacher is NEVER silently invented or retrained.')
    ck=torch_load(candidate)
    if ck.get('arch')!='a3' or ck.get('method')!='lora' or 'model' not in ck:
        raise RuntimeError('Teacher is not a trained Phase6 A3 LoRA best.pt checkpoint')
    meta=ck.get('settings',{})
    if meta.get('tokenizer_sha')!=backbone_bundle['tokenizer_sha']:
        raise RuntimeError('Teacher/student Phase1 tokenizer SHA-256 mismatch')
    cfg=ck.get('model_config',backbone_bundle['config'])
    if cfg!=backbone_bundle['config']:raise RuntimeError('Teacher Phase1 architecture config differs from student Phase1 config')
    teacher=create_model('a3',len(tokenizer),cfg)
    t_rank=int(meta.get('rank',8));t_alpha=int(meta.get('alpha',16))
    adapt_model(teacher,'a3','lora',rank=t_rank,alpha=t_alpha)
    teacher.load_state_dict(ck['model'],strict=True)
    for p in teacher.parameters():p.requires_grad=False
    teacher.eval().to(device)
    return teacher,str(Path(candidate).resolve())

def evaluate_dev(model,arch,tokenizer,rows,args,device,log_path=None):
    model.eval()
    refs=[];hyps=[];samples=[];failure=[]
    subset=rows[:args.dev_eval_max]
    with torch.inference_mode():
        for i,row in enumerate(subset):
            try:
                with amp_context(device):
                    txt=generate(model,arch,tokenizer,row['document'],args.max_source,args.decode_length,args.strategy,device=device)
                refs.append(row['summary']);hyps.append(txt)
                if len(samples)<6:samples.append({'id':row['id'],'reference':row['summary'][:1800],'prediction':txt[:1800]})
            except (ValueError,FloatingPointError,RuntimeError) as e:
                # Keep full evaluation denominator; don't silently score only easy examples.
                failure.append({'id':row['id'],'error':str(e)})
                refs.append(row['summary']);hyps.append('')
        metrics=evaluate_pairs(refs,hyps)
        metrics['empty_or_failed_generations']=len(failure)
        metrics['bertscore_f1']=optional_bertscore(refs,hyps,args.bertscore)
    if log_path:save_json(log_path,{'metrics':metrics,'examples':samples,'failures':failure[:100]})
    model.train()
    return metrics

def main():
    a=parse_args();validate_args(a)
    if not a.cpu_test and not torch.cuda.is_available():raise RuntimeError('GPU unavailable. Enable a Kaggle T4 GPU before running real training.')
    device=torch.device('cpu' if a.cpu_test else 'cuda')
    set_seed(a.seed)
    out=Path(a.out).resolve();out.mkdir(parents=True,exist_ok=True)
    token=get_hf_token(required=False)
    all_data,datameta=read_data('train',a.data_dir,token)
    train,dev,audit=partition_train(all_data,a.seed,a.dev_fraction)
    audit['dataset']=datameta
    save_json(out/'data_audit.json',audit)
    bundle=prepare_phase1(a.arch,a.phase1_repo,a.phase1_dir,a.phase1_weight,token)
    tokenizer=load_tokenizer(bundle)
    if len(tokenizer)!=bundle['checkpoint_vocab']:
        raise RuntimeError(f'Phase1 tokenizer vocab size {len(tokenizer)} differs from checkpoint vocab size {bundle["checkpoint_vocab"]}')
    model=create_model(a.arch,len(tokenizer),bundle['config'],a.student_layers if a.method=='kd' else 0)
    load_phase1_weights(model,bundle['weights'],a.student_layers if a.method=='kd' else 0)
    paraminfo=adapt_model(model,a.arch,a.method,a.rank,a.alpha,a.adapter_size)
    if a.lr is None:a.lr=2e-5 if a.method=='full' else (5e-5 if a.method=='partial' else 2e-4)
    teacher=None;teacher_file=None
    if a.method=='kd':teacher,teacher_file=load_teacher(a,token,bundle,tokenizer,device)
    model.to(device)
    optimizer=torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),lr=a.lr,weight_decay=a.weight_decay)
    scaler=torch.amp.GradScaler('cuda',enabled=device.type=='cuda')
    settings={'dataset_sha':datameta['sha256'],'tokenizer_sha':bundle['tokenizer_sha'],'phase1_weight_sha':bundle['weight_sha'],
              'model_config':bundle['config'],'arch':a.arch,'method':a.method,'rank':a.rank,'alpha':a.alpha,
              'adapter_size':a.adapter_size,'student_layers':a.student_layers,'teacher':teacher_file,
              'seed':a.seed,'dev_fraction':a.dev_fraction,'max_source':a.max_source,'max_target':a.max_target,
              'strategy':a.strategy,'accum':a.accum,'lr':a.lr,'weight_decay':a.weight_decay,
              'kd_temperature':a.kd_temperature,'kd_soft_weight':a.kd_soft_weight}
    save_json(out/'run_config.json',dict(vars(a),phase1={'source':bundle['source'],'weight_relative':bundle['weight_relative'],
               'tokenizer_sha':bundle['tokenizer_sha'],'weight_sha':bundle['weight_sha'],'config_source':bundle['config_source']},paraminfo=paraminfo))
    start_epoch=0;cursor=0;global_step=0;best=-float('inf');best_metric='BERTScore F1' if a.bertscore else 'ROUGE-L F1 (local proxy)'
    last=out/'last.pt'
    if last.is_file() and not a.no_resume:
        ck=torch_load(last);start_epoch,cursor,global_step=restore_training_checkpoint(ck,model,optimizer,a.arch,a.method,settings)
        best=float(ck.get('best_score',-float('inf')))
        print(f'Resumed from epoch {start_epoch}, example {cursor}, optimizer-step {global_step}',flush=True)
    print('Architecture:',a.arch,'Method:',a.method,'Params:',paraminfo,'Train:',len(train),'Dev:',len(dev),'Device:',device,flush=True)
    print('Primary official metric: BERTScore F1. Local checkpoint metric:',best_metric,flush=True)
    logs=out/'training_log.jsonl'
    for epoch in range(start_epoch,a.epochs):
        order=list(range(len(train)));random.Random(a.seed+epoch).shuffle(order)
        model.train();optimizer.zero_grad(set_to_none=True)
        elapsed=time.time();loss_sum=0.;count=0
        first=cursor if epoch==start_epoch else 0
        for i in range(first,len(order)):
            row=train[order[i]]
            batch=batch_tokens(tokenizer,row,a.arch,a.max_source,a.max_target,a.strategy)
            batch={k:v.to(device) for k,v in batch.items()}
            try:
                with amp_context(device):
                    if teacher is None:loss=target_loss(model,a.arch,batch,tokenizer)
                    else:
                        hard,stu_logits,mask=target_loss(model,a.arch,batch,tokenizer,return_logits=True)
                        with torch.no_grad():
                            _,teacher_logits,tmask=target_loss(teacher,'a3',batch,tokenizer,return_logits=True)
                        if not torch.equal(mask,tmask):raise RuntimeError('KD teacher/student label masks are misaligned')
                        T=a.kd_temperature
                        s=stu_logits[mask].float();t=teacher_logits[mask].float()
                        soft=F.kl_div(F.log_softmax(s/T,dim=-1),F.softmax(t/T,dim=-1),reduction='batchmean')*(T*T)
                        loss=(1-a.kd_soft_weight)*hard+a.kd_soft_weight*soft
                    if not torch.isfinite(loss):raise FloatingPointError('Non-finite model loss')
                    scaled_loss=loss/a.accum
                scaler.scale(scaled_loss).backward()
                loss_sum+=float(loss.detach().cpu());count+=1
                if (i+1)%a.accum==0 or i==len(order)-1:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad),1.0,error_if_nonfinite=True)
                    scaler.step(optimizer);scaler.update();optimizer.zero_grad(set_to_none=True)
                    global_step+=1
                    if global_step%a.save_every==0:
                        cp=make_training_checkpoint(model,optimizer,a.arch,a.method,settings,epoch,i+1,global_step)
                        cp['best_score']=best;atomic_torch_save(cp,last)
            except torch.cuda.OutOfMemoryError as e:
                optimizer.zero_grad(set_to_none=True);torch.cuda.empty_cache()
                raise RuntimeError(f'CUDA OOM at epoch={epoch} example={i}. Reduce --max-source or --max-target; a checkpoint may exist in {last}') from e
            except Exception as e:
                raise RuntimeError(f'Phase6 training failed arch={a.arch} method={a.method} epoch={epoch} idx={i} id={row["id"]}: {e}') from e
        metrics=evaluate_dev(model,a.arch,tokenizer,dev,a,device,out/f'dev_epoch_{epoch+1}.json')
        score=metrics['bertscore_f1'] if a.bertscore else metrics['rougeL_f1']
        log={'epoch':epoch+1,'step':global_step,'train_loss':round(loss_sum/max(count,1),6),'dev':metrics,'seconds':round(time.time()-elapsed,1)}
        store_event(logs,log);print(json.dumps(log),flush=True)
        if score>best:
            best=score
            cp=make_training_checkpoint(model,optimizer,a.arch,a.method,settings,epoch+1,0,global_step,best)
            cp['model_config']=bundle['config'];cp['best_score']=best;cp['selected_metric']=best_metric
            cp['teacher_checkpoint']=teacher_file
            cp['optimizer']=None  # Best inference checkpoint is compact; last.pt retains optimizer for resumption.
            atomic_torch_save(cp,out/'best.pt')
            save_json(out/'best_metrics.json',{'score':best,'metric':best_metric,'epoch':epoch+1,'metrics':metrics,'trainable':paraminfo})
        cp=make_training_checkpoint(model,optimizer,a.arch,a.method,settings,epoch+1,0,global_step,best)
        cp['best_score']=best;atomic_torch_save(cp,last)
        cursor=0
    if not (out/'best.pt').is_file():raise RuntimeError('Training finished without best.pt; cannot generate predictions')
    print('Saved best:',out/'best.pt','metric',best_metric,best,flush=True)

if __name__=='__main__':
    try:main()
    except Exception:
        traceback.print_exc();sys.exit(1)


In [ ]:
%%writefile phase6_predict.py
"""Phase 6 prediction (the Phase 5 phase5_predict.py workflow).
  python phase6_predict.py --run runs/a1_full --out submissions/a1_full
Regenerates legal summaries; unlike Phase 5, no label thresholds or statute IDs.
Checks Phase1 checkpoint/tokenizer, resumes partial inference and validates ZIP.
"""
import argparse, contextlib, json, os, sys, traceback
from pathlib import Path
import torch
from phase6_core import *

def parse_args():
    p=argparse.ArgumentParser()
    p.add_argument('--run',default='',help='Phase 5 style: experiment folder containing best.pt')
    p.add_argument('--checkpoint',default='',help='Direct path to trained Phase6 best.pt')
    p.add_argument('--out',required=True,help='submissions/<method>')
    p.add_argument('--data-dir',default='')
    p.add_argument('--phase1-repo',default='')
    p.add_argument('--phase1-dir',default='')
    p.add_argument('--phase1-weight',default='')
    p.add_argument('--decode-length',type=int,default=112)
    p.add_argument('--max-source',type=int,default=448)
    p.add_argument('--strategy',choices=['head_tail','head_middle_tail'],default='head_middle_tail')
    p.add_argument('--save-every',type=int,default=10)
    p.add_argument('--no-repeat-ngram',type=int,default=0,
                   help='0=original greedy; 4 blocks repeated 4-token phrases')
    p.add_argument('--cpu-test',action='store_true')
    return p.parse_args()

def main():
    a=parse_args()
    if a.save_every<1 or a.decode_length<1:raise ValueError('save-every/decode-length must be positive')
    if a.no_repeat_ngram != 0 and not 2 <= a.no_repeat_ngram <= a.decode_length:
        raise ValueError('--no-repeat-ngram must be 0 or between 2 and decode-length')
    if not a.cpu_test and not torch.cuda.is_available():raise RuntimeError('Enable GPU for test inference')
    device=torch.device('cpu' if a.cpu_test else 'cuda');token=get_hf_token()
    if bool(a.run)==bool(a.checkpoint):
        raise ValueError('Supply ONE of --run runs/<method> OR --checkpoint path/to/best.pt')
    ckfile=(Path(a.run)/'best.pt' if a.run else Path(a.checkpoint)).resolve()
    if not ckfile.is_file():raise FileNotFoundError(f'Trained Phase6 checkpoint is missing: {ckfile}')
    ck=torch_load(ckfile)
    if ck.get('version')!=1 or ck.get('method') not in ('full','partial','lora','lorma','adapters','kd'):
        raise ValueError('Not a Phase6 best.pt training checkpoint')
    arch=ck['arch'];method=ck['method'];settings=ck['settings']
    bundle=prepare_phase1(arch,a.phase1_repo,a.phase1_dir,a.phase1_weight,token)
    if bundle['tokenizer_sha']!=settings.get('tokenizer_sha'):
        raise RuntimeError('Tokenizer mismatch with trained Phase6 checkpoint. Cannot safely generate.')
    if bundle['weight_sha']!=settings.get('phase1_weight_sha'):
        raise RuntimeError('Phase1 source weights do not match training checkpoint')
    if bundle['config']!=settings['model_config']:
        raise RuntimeError('Phase1 model configuration mismatch with trained Phase6 checkpoint')
    tokenizer=load_tokenizer(bundle)
    if len(tokenizer)!=bundle['checkpoint_vocab']:
        raise RuntimeError(f'Phase1 tokenizer vocab size {len(tokenizer)} differs from checkpoint vocab size {bundle["checkpoint_vocab"]}')
    student_layers=int(settings.get('student_layers',0)) if method=='kd' else 0
    model=create_model(arch,len(tokenizer),bundle['config'],student_layers)
    adapt_model(model,arch,method,int(settings.get('rank',8)),int(settings.get('alpha',16)),int(settings.get('adapter_size',32)))
    model.load_state_dict(ck['model'],strict=True)
    model.eval().to(device)
    test,datameta=read_data('test',a.data_dir,token)
    outdir=Path(a.out).resolve();outdir.mkdir(parents=True,exist_ok=True)
    model_sha=sha_file(ckfile)
    progress=outdir/'prediction_progress.json'
    done=[]
    if progress.is_file():
        p=read_json(progress)
        if p.get('test_sha')!=datameta['sha256'] or p.get('model_sha')!=model_sha or p.get('decode_length')!=a.decode_length or p.get('max_source')!=a.max_source or p.get('strategy')!=a.strategy or p.get('no_repeat_ngram', 0)!=a.no_repeat_ngram:
            raise RuntimeError('Existing prediction_progress.json belongs to another model/test/settings. Move it aside to start fresh.')
        done=p.get('predictions',[])
        if len(done)>len(test) or any(not isinstance(r,dict) or r.get('id')!=test[i]['id'] or not isinstance(r.get('summary'),str) or not r['summary'].strip() for i,r in enumerate(done)):
            raise RuntimeError('Corrupt/out-of-order prediction progress file')
    def checkpoint_progress():
        save_json(progress,{'test_sha':datameta['sha256'],'model_sha':model_sha,'decode_length':a.decode_length,
              'max_source':a.max_source,'strategy':a.strategy,
              'no_repeat_ngram':a.no_repeat_ngram,'predictions':done})
    print('Inference:',method,arch,'completed:',len(done),'remaining:',len(test)-len(done),'test_hash:',datameta['sha256'][:16],flush=True)
    for i in range(len(done),len(test)):
        row=test[i]
        try:
            with torch.inference_mode():
                ctx=torch.autocast('cuda',dtype=torch.float16) if device.type=='cuda' else contextlib.nullcontext()
                with ctx:
                    summary=generate(model,arch,tokenizer,row['document'],a.max_source,a.decode_length,a.strategy,
                                     device=device,no_repeat_ngram_size=a.no_repeat_ngram)
            done.append({'id':row['id'],'summary':summary})
        except torch.cuda.OutOfMemoryError as e:
            checkpoint_progress();torch.cuda.empty_cache()
            raise RuntimeError(f'Inference CUDA OOM for ID={row["id"]}, saved {len(done)} predictions. Reduce --max-source.') from e
        except Exception as e:
            checkpoint_progress()
            raise RuntimeError(f'Inference failed for ID={row["id"]}, saved {len(done)} complete predictions: {e}') from e
        if len(done)%a.save_every==0:checkpoint_progress();print('Completed:',len(done),'/',len(test),flush=True)
    checkpoint_progress()
    pred,zipfile=package_submission(done,test,outdir)
    save_json(outdir/'inference_audit.json',{'method':method,'arch':arch,'phase1_source':bundle['source'],'phase1_weight':bundle['weight_relative'],
      'test':datameta,'checkpoint_sha256':model_sha,'prediction_sha256':sha_file(pred),
      'zip_sha256':sha_file(zipfile),'n_examples':len(done),
      'no_repeat_ngram':a.no_repeat_ngram,
      'repetition_audit':repetition_summary([r['summary'] for r in done])})
    audit=read_json(outdir/'inference_audit.json')['repetition_audit']
    print('VALIDATED PREDICTIONS:',pred,'ZIP:',zipfile,flush=True)
    print('Repetition audit:',audit,flush=True)
    if audit['high_repetition_count']:
        print('WARNING: Repeated word phrases remain in some summaries; inspect DEV examples and compare scores.',flush=True)

if __name__=='__main__':
    try:main()
    except Exception:traceback.print_exc();sys.exit(1)


In [ ]:
%%writefile phase6_validate.py
"""Phase 6 validator (Phase 5 phase5_validate.py role).
  python phase6_validate.py --pred submissions/a1_full/submission.zip
Checks official IDs, nonempty generated summary strings and the ZIP root.
It does not invent hidden test scores; BERTScore/ROUGE belong to dev analysis.
"""
import argparse,json,sys,traceback,zipfile
from pathlib import Path
from phase6_core import read_data,validate_submission,sha_file,save_json,read_json

def main():
    p=argparse.ArgumentParser();p.add_argument('--pred','--zip',dest='pred',required=True,help='prediction ZIP (Phase5 style --pred)');p.add_argument('--data-dir',default='');p.add_argument('--audit-out',default='');a=p.parse_args()
    archive=Path(a.pred)
    if not archive.is_file():raise FileNotFoundError(archive)
    if not zipfile.is_zipfile(archive):raise ValueError('Not a ZIP')
    with zipfile.ZipFile(archive) as z:
        names=z.namelist()
        if names!=['prediction.json']:raise ValueError(f'ZIP root must contain ONLY prediction.json, got {names}')
        if z.testzip() is not None:raise ValueError('Corrupt ZIP payload')
        pred=json.loads(z.read('prediction.json').decode('utf-8'))
    test,meta=read_data('test',a.data_dir)
    report=validate_submission(pred,test)
    report['zip_sha256']=sha_file(archive);report['official_test_sha256']=meta['sha256']
    if a.audit_out:save_json(a.audit_out,report)
    print(json.dumps({'PASS':True,**report},indent=2))

if __name__=='__main__':
    try:main()
    except Exception:traceback.print_exc();sys.exit(1)


In [ ]:
%%writefile phase6_evaluate.py
"""Phase 6 local held-out DEV metrics; NOT Phase 5 statute scoring.

Usage:
  python phase6_evaluate.py --run runs/a1_full --limit 64 --phase1-repo USER/REPO

Generates summaries with the trained best.pt checkpoint and then measures
ROUGE-1/L + real BERTScore P/R/F1 using RoBERTa-large (evaluation ONLY).
The official private Codabench scores cannot be reproduced exactly without
its private references and exact scorer settings.
"""
import argparse, contextlib, gc, json, sys, traceback
from pathlib import Path
import torch
from phase6_core import (get_hf_token, read_data, partition_train, prepare_phase1,
    load_tokenizer, create_model, adapt_model, torch_load, generate,
    evaluate_pairs, save_json, read_json, sha_file)


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--run', required=True, help='Experiment folder containing best.pt')
    p.add_argument('--phase1-repo', default='')
    p.add_argument('--phase1-dir', default='')
    p.add_argument('--phase1-weight', default='')
    p.add_argument('--data-dir', default='')
    p.add_argument('--limit', type=int, default=64, help='Held-out DEV subset, not test')
    p.add_argument('--no-repeat-ngram', type=int, default=0,
                   help='0=original greedy; 4 blocks repeated 4-token phrases')
    p.add_argument('--model-type', default='roberta-large', help='BERTScore evaluation model')
    p.add_argument('--batch-size', type=int, default=2, help='Small to limit evaluator GPU memory')
    p.add_argument('--device', choices=['auto', 'cuda', 'cpu'], default='auto')
    a = p.parse_args()
    if a.no_repeat_ngram != 0 and a.no_repeat_ngram < 2:
        raise ValueError('--no-repeat-ngram must be 0 or >=2')
    if a.limit < 1 or a.batch_size < 1:
        raise ValueError('--limit and --batch-size must be positive')
    if a.device == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA requested for local scoring but no GPU is available')
    device = 'cuda' if a.device == 'auto' and torch.cuda.is_available() else a.device
    if device == 'auto': device = 'cpu'
    run = Path(a.run).resolve()
    weight = run / 'best.pt'
    if not weight.is_file():
        raise FileNotFoundError(f'Missing trained Phase 6 checkpoint: {weight}')
    ck = torch_load(weight)
    if ck.get('version') != 1 or ck.get('method') not in ('full', 'partial', 'lora', 'lorma', 'adapters', 'kd'):
        raise ValueError('This is not a supported trained Phase 6 best.pt')
    s = ck['settings']; arch, method = ck['arch'], ck['method']
    token = get_hf_token()
    examples, dataset = read_data('train', a.data_dir, token)
    if dataset['sha256'] != s['dataset_sha']:
        raise RuntimeError('Dataset checksum differs from training. Refusing to score a different DEV split.')
    _, dev, _ = partition_train(examples, int(s['seed']), float(s['dev_fraction']))
    chosen = dev[:a.limit]
    run_cfg = read_json(run / 'run_config.json') if (run / 'run_config.json').is_file() else {}
    decode_length = int(run_cfg.get('decode_length', 112))
    if decode_length < 1:
        raise ValueError('Saved decode length must be positive')
    if a.no_repeat_ngram > decode_length:
        raise ValueError('--no-repeat-ngram cannot exceed saved decode-length')
    bundle = prepare_phase1(arch, a.phase1_repo, a.phase1_dir, a.phase1_weight, token)
    if bundle['tokenizer_sha'] != s['tokenizer_sha'] or bundle['weight_sha'] != s['phase1_weight_sha']:
        raise RuntimeError('Phase 1 weight/tokenizer hashes do not match the trained Phase 6 checkpoint')
    if bundle['config'] != s['model_config']:
        raise RuntimeError('Phase 1 architecture config changed from training')
    tok = load_tokenizer(bundle)
    if len(tok) != bundle['checkpoint_vocab']:
        raise RuntimeError('Tokenizer vocabulary size does not match Phase 1 checkpoint')
    layers = int(s.get('student_layers', 0)) if method == 'kd' else 0
    model = create_model(arch, len(tok), bundle['config'], layers)
    adapt_model(model, arch, method, int(s.get('rank', 8)), int(s.get('alpha', 16)), int(s.get('adapter_size', 32)))
    model.load_state_dict(ck['model'], strict=True)
    model.eval().to(device)

    # Resume DEV generation only when checkpoint and generation settings match exactly.
    suffix = f'_nr{a.no_repeat_ngram}' if a.no_repeat_ngram else ''
    progress = run / f'dev_bertscore_progress{suffix}.json'
    identity = dict(checkpoint_sha=sha_file(weight), dataset_sha=dataset['sha256'],
                    ids=[r['id'] for r in chosen], max_source=int(s['max_source']),
                    decode_length=decode_length, strategy=s['strategy'],
                    no_repeat_ngram=a.no_repeat_ngram)
    pairs = []
    if progress.exists():
        old = read_json(progress)
        if any(old.get(k) != v for k, v in identity.items()):
            raise RuntimeError(f'{progress} belongs to a different checkpoint/DEV selection; move it aside to start fresh')
        pairs = old.get('pairs', [])
        if len(pairs) > len(chosen) or any(not isinstance(row, dict) or
                row.get('id') != chosen[i]['id'] or row.get('reference') != chosen[i]['summary'] or
                not isinstance(row.get('prediction'), str) or not row['prediction'].strip()
                for i, row in enumerate(pairs)):
            raise ValueError('Invalid or out-of-order DEV evaluation progress file')
    print(f'Local DEV generation: {len(pairs)}/{len(chosen)} complete; {arch}/{method}', flush=True)
    for i in range(len(pairs), len(chosen)):
        row = chosen[i]
        try:
            with torch.inference_mode():
                ctx = torch.autocast('cuda', dtype=torch.float16) if device == 'cuda' else contextlib.nullcontext()
                with ctx:
                    hypothesis = generate(model, arch, tok, row['document'],
                        int(s['max_source']), decode_length,
                        s['strategy'], device=torch.device(device),
                        no_repeat_ngram_size=a.no_repeat_ngram)
            pairs.append({'id': row['id'], 'reference': row['summary'], 'prediction': hypothesis})
            if len(pairs) % 8 == 0:
                save_json(progress, {**identity, 'pairs': pairs})
        except Exception as exc:
            save_json(progress, {**identity, 'pairs': pairs})
            raise RuntimeError(f'DEV generation stopped at id={row["id"]}, saved {len(pairs)} examples: {exc}') from exc
    save_json(progress, {**identity, 'pairs': pairs})
    refs = [r['reference'] for r in pairs]
    hyps = [r['prediction'] for r in pairs]
    metrics = evaluate_pairs(refs, hyps)
    del model, ck
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    # The scorer loads an external RoBERTa model solely for local metric calculation.
    # It is never loaded into or used to train/generate from the student's model.
    try:
        from bert_score import BERTScorer
    except ImportError as exc:
        raise RuntimeError('Install the separate metric package: %pip install bert-score==0.3.13') from exc
    try:
        scorer = BERTScorer(model_type=a.model_type, lang='en', device=device,
                            batch_size=a.batch_size, idf=False, rescale_with_baseline=False)
        precision, recall, f1 = scorer.score(hyps, refs, batch_size=a.batch_size)
    except Exception as exc:
        raise RuntimeError(f'BERTScore evaluation failed ({a.model_type}, {device}). '
            'Ensure Kaggle Internet is enabled to download the metric model; '
            'if GPU memory is insufficient rerun with --device cpu. '
            f'Original error: {exc}') from exc
    if not all(torch.isfinite(t).all() for t in (precision, recall, f1)):
        raise FloatingPointError('BERTScore produced nonfinite values; no metric file saved')
    metrics.update(bertscore_precision=round(float(precision.mean()), 6),
                   bertscore_recall=round(float(recall.mean()), 6),
                   bertscore_f1=round(float(f1.mean()), 6))
    record = {'split': 'held_out_train_dev', 'metrics': metrics, 'n_dev_total': len(dev),
              'n_scored': len(chosen), 'architecture': arch, 'method': method,
              'no_repeat_ngram': a.no_repeat_ngram,
              'checkpoint_sha256': identity['checkpoint_sha'],
              'checkpoint_selected_by': read_json(run / 'best_metrics.json').get('metric') if (run / 'best_metrics.json').is_file() else 'unknown',
              'scorer': {'model_type': a.model_type, 'hash': scorer.hash,
                         'idf': False, 'rescale_with_baseline': False, 'device': device}}
    save_json(run / f'local_dev_bertscore{suffix}.json', record)
    print('LOCAL DEV SCORES (not private Codabench test scores):', json.dumps(record, indent=2), flush=True)

if __name__ == '__main__':
    try: main()
    except Exception: traceback.print_exc(); sys.exit(1)


In [ ]:
%%writefile phase6_hf.py
"""Private HF backup/recovery — Phase 5's save-to-HF step, reused in Phase 6.
Only your private experiment repository is updated; your Phase1 repos are read-only.
"""
from pathlib import Path
import json
from huggingface_hub import HfApi, snapshot_download
from huggingface_hub.errors import RepositoryNotFoundError

# Upload all user-created workspace files, not Hugging Face's internal cache or secrets.
IGNORED = ['.cache/**', '**/.cache/**', '**/__pycache__/**',
           '**/.ipynb_checkpoints/**', '**/*.tmp', '**/*.lock',
           '**/.env', '**/.env.*', '**/*secret*']

def connect(token, repo_name):
    if not token:
        raise RuntimeError('Add your Hugging Face write token as Kaggle Secret HF.')
    api = HfApi(token=token)
    username = api.whoami()['name']  # Account is discovered, never hardcoded.
    repo_id = f'{username}/{repo_name}'
    try:
        files = api.list_repo_files(repo_id=repo_id, repo_type='model')
    except RepositoryNotFoundError:
        files = []
    api.create_repo(repo_id=repo_id, private=True, exist_ok=True)
    if not api.repo_info(repo_id=repo_id).private:
        raise RuntimeError('Refusing backup: the destination repository is not private.')
    return api, username, repo_id, files

def restore(token, repo_id, workspace, known_files):
    if not known_files:
        print('New HF repository: no previous run to restore.')
        return
    # Restore saved checkpoints/partial generations, not old source code.
    patterns = ['runs/**', 'submissions/**', 'phase1_cache/**']
    if any(f.startswith(('runs/','submissions/','phase1_cache/')) for f in known_files):
        snapshot_download(repo_id=repo_id, repo_type='model', token=token,
                          local_dir=str(workspace), allow_patterns=patterns)
        print('Restored existing checkpoints and predictions from:', repo_id)

def backup(api, repo_id, workspace, required=()):
    root = Path(workspace).resolve()
    for rel in required:
        if not (root / rel).is_file():
            raise FileNotFoundError(f'Cannot back up incomplete experiment. Missing: {rel}')
    manifest = {'repo': repo_id, 'workspace': root.name, 'required': list(required)}
    (root / 'hf_backup_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    # Upload the entire experiment WORKDIR to the private repo root, updating existing paths.
    api.upload_folder(repo_id=repo_id, repo_type='model', folder_path=str(root),
                      path_in_repo='', ignore_patterns=IGNORED,
                      commit_message='Update full Phase 6 Kaggle workspace')
    remote = set(api.list_repo_files(repo_id=repo_id, repo_type='model'))
    # Check all regular workspace files, not only the checkpoint/ZIP.
    expected = set(required)
    for path in root.rglob('*'):
        if not path.is_file() or path.is_symlink():
            continue
        rel = path.relative_to(root)
        if any(part in ('.cache', '__pycache__', '.ipynb_checkpoints') for part in rel.parts):
            continue
        if (path.suffix in ('.tmp', '.lock') or path.name.startswith('.env')
                or 'secret' in path.name.lower()):
            continue
        expected.add(rel.as_posix())
    missing = expected - remote
    if missing:
        raise RuntimeError(f'HF upload verification failed, missing remote files: {sorted(missing)}')
    print('Verified private HF workspace backup:', 'https://huggingface.co/' + repo_id)
    print('Verified workspace files:', len(expected))
    return {'repo':repo_id, 'verified':True, 'remote_file_count':len(remote)}


## Run the experiment — Phase 5 order

HF login → recover checkpoint → choose settings → verify Phase 1 → load official data → train → predict → validate → back up.
Each action is a short separate Kaggle cell. Stop if a check raises an error.

In [ ]:
# 3. Authenticate with Kaggle Secret HF; detect your username automatically.
from huggingface_hub import HfApi
from phase6_core import get_hf_token, DEFAULT_REPOS
from phase6_hf import connect, restore, backup
HF_TOKEN = get_hf_token(required=True)
HF_API, HF_USER, REPO_ID, REMOTE_FILES = connect(HF_TOKEN, 'cs781-phase6-a1-full')
print('HF account:', HF_USER)
print('Private output repository:', REPO_ID)

In [ ]:
# 4. Restore previous checkpoints if this experiment ran in an earlier Kaggle session.
# The model source cells above remain the current version.
restore(HF_TOKEN, REPO_ID, WORKDIR, REMOTE_FILES)

In [ ]:
# 5. Configure the experiment. Phase 1 repo may be changed if your account uses another name.
ARCH, METHOD, SLUG = 'a1', 'full', 'a1_full'
PHASE1_REPO = os.getenv('PHASE1_REPO_'+ARCH.upper(), DEFAULT_REPOS[ARCH])  # Original Phase 1 source; override if different.
PHASE1_DIR = ''         # Optional mounted local Phase 1 weights/tokenizer.
PHASE1_WEIGHT = ''      # Optional explicit checkpoint filename.
DATA_DIR = ''           # Empty: read official Phase_6_Summ from HF.
EPOCHS, ACCUM, SEED = 3, 4, 44
MAX_SOURCE, MAX_TARGET, DECODE_LENGTH = 448, 112, 112
DEV_EVAL_MAX = 64       # Only local held-out evaluation, never private test labels.
LOCAL_BERTSCORE = False # False: fast ROUGE-L checkpoint selection. True: slower BERTScore per epoch.
RUN_LOCAL_BERTSCORE = True  # Score your best checkpoint with real BERTScore on held-out DEV.
SCORER_DEVICE = 'auto'     # Choose 'cpu' if the RoBERTa-large scorer runs out of GPU memory.
OUT_DIR = WORKDIR / 'runs' / SLUG
# Run a new decoding variant; keep existing predictions/checkpoints unchanged.
NO_REPEAT_NGRAM = 4  # Set 0 for the original greedy generation.
PRED_SLUG = f'{SLUG}_nr{NO_REPEAT_NGRAM}' if NO_REPEAT_NGRAM else SLUG
PRED_DIR = WORKDIR / 'submissions' / PRED_SLUG

def run(*args):
    # Raise immediately on a subprocess failure instead of continuing silently.
    subprocess.run([sys.executable, *map(str,args)], check=True)

COMMON = ['--epochs',EPOCHS,'--batch-size',1,'--accum',ACCUM,
          '--max-source',MAX_SOURCE,'--max-target',MAX_TARGET,
          '--decode-length',DECODE_LENGTH,'--dev-eval-max',DEV_EVAL_MAX,
          '--seed',SEED,'--phase1-repo',PHASE1_REPO]
if PHASE1_DIR: COMMON += ['--phase1-dir', PHASE1_DIR]
if PHASE1_WEIGHT: COMMON += ['--phase1-weight', PHASE1_WEIGHT]
if DATA_DIR: COMMON += ['--data-dir', DATA_DIR]
if LOCAL_BERTSCORE: COMMON += ['--bertscore']
print('Phase1 repo:', PHASE1_REPO)

In [ ]:
# 6. Fetch the exact original Phase 1 model and tokenizer (fail if missing/mismatched).
from phase6_core import prepare_phase1, load_tokenizer
bundle = prepare_phase1(ARCH, PHASE1_REPO, PHASE1_DIR, PHASE1_WEIGHT, HF_TOKEN)
tokenizer = load_tokenizer(bundle)
if len(tokenizer) != bundle['checkpoint_vocab']:
    raise RuntimeError('Tokenizer and Phase 1 checkpoint vocabulary sizes differ.')
print('Phase 1 weights:', bundle['weight_relative'])
print('Tokenizer verified; vocab size:', len(tokenizer))

In [ ]:
# 7. Locate the official train and test files. No private references are needed.
from phase6_core import locate_dataset_file
print('Train:', locate_dataset_file('train', DATA_DIR, HF_TOKEN))
print('Test: ', locate_dataset_file('test', DATA_DIR, HF_TOKEN))

In [ ]:
# 8. Train or resume this model on official Phase 6 training documents.
# Checkpoints and dev metrics are stored under runs/<method>/.
run('phase6_train.py', '--arch', ARCH, '--method', METHOD,
    '--out', OUT_DIR, *COMMON)

In [ ]:
# 9. Inspect the best *held-out DEV* result; never use private test labels.
from phase6_core import read_json
import json
best_metrics = OUT_DIR / 'best_metrics.json'
if not best_metrics.is_file():
    raise FileNotFoundError(f'Missing validation metrics: {best_metrics}')
print(json.dumps(read_json(best_metrics), indent=2))

In [ ]:
# 10. Back up the complete workspace after training.
backup(HF_API, REPO_ID, WORKDIR,
       required=[f'runs/{SLUG}/best.pt'])

In [ ]:
# 11. Generate predictions from the trained best checkpoint (resume safe).
args = ['phase6_predict.py', '--run', OUT_DIR,
        '--out', PRED_DIR, '--decode-length', DECODE_LENGTH,
        '--max-source', MAX_SOURCE,
        '--no-repeat-ngram', NO_REPEAT_NGRAM, '--phase1-repo', PHASE1_REPO]
if PHASE1_DIR: args += ['--phase1-dir', PHASE1_DIR]
if PHASE1_WEIGHT: args += ['--phase1-weight', PHASE1_WEIGHT]
if DATA_DIR: args += ['--data-dir', DATA_DIR]
run(*args)

In [ ]:
# 12. Validate IDs, summary strings and ZIP structure locally.
args = ['phase6_validate.py', '--pred', PRED_DIR/'submission.zip']
if DATA_DIR: args += ['--data-dir', DATA_DIR]
run(*args)

In [ ]:
# 13. Show one short generated example after successful validation.
from phase6_core import read_json
predictions = read_json(PRED_DIR / 'prediction.json')
print('Test documents:', len(predictions))
print('First summary:', predictions[0]['summary'][:400])

In [ ]:
# 14. Upload EVERY file in this experiment WORKDIR and verify remote artifacts.
backup(HF_API, REPO_ID, WORKDIR, required=[
    f'phase6_backbones/{ARCH}.py',
    f'runs/{SLUG}/best.pt',
    f'submissions/{PRED_SLUG}/prediction.json',
    f'submissions/{PRED_SLUG}/submission.zip'])

In [ ]:
# Extra: compute real local BERTScore on the trained best checkpoint.
# The official test prediction ZIP has already been validated AND backed up.
# The scoring model is used ONLY for reference-based evaluation, never for training.
if RUN_LOCAL_BERTSCORE:
    args = ['phase6_evaluate.py', '--run', OUT_DIR, '--no-repeat-ngram', NO_REPEAT_NGRAM, '--limit', DEV_EVAL_MAX,
            '--batch-size', 2, '--device', SCORER_DEVICE, '--phase1-repo', PHASE1_REPO]
    if PHASE1_DIR: args += ['--phase1-dir', PHASE1_DIR]
    if PHASE1_WEIGHT: args += ['--phase1-weight', PHASE1_WEIGHT]
    if DATA_DIR: args += ['--data-dir', DATA_DIR]
    run(*args)
    metric_file = OUT_DIR / f'local_dev_bertscore_nr{NO_REPEAT_NGRAM}.json' if NO_REPEAT_NGRAM else OUT_DIR / 'local_dev_bertscore.json'
    new_score = read_json(metric_file)['metrics']
    print('Local DEV BERTScore with current decoding:', new_score)
    # Compare against the old generation method if the previous metric exists.
    previous = OUT_DIR / 'local_dev_bertscore.json'
    if previous != metric_file and previous.is_file():
        old_score = read_json(previous)['metrics']
        print('Previous greedy BERTScore F1:', old_score['bertscore_f1'],
              '| new decoding BERTScore F1:', new_score['bertscore_f1'])
        print('Compare ROUGE and actual summaries as well before submitting.')
    # Back up new metric result and resumable evaluation progress.
    backup(HF_API, REPO_ID, WORKDIR, required=[str(metric_file.relative_to(WORKDIR))])


In [ ]:
# 15. Show final files, private backup and what must be submitted.
print('HF repo:', 'https://huggingface.co/' + REPO_ID)
print('Checkpoint:', OUT_DIR/'best.pt')
print('Prediction JSON:', PRED_DIR/'prediction.json')
print('Codabench ZIP:', PRED_DIR/'submission.zip')
print('Submit the ZIP yourself; private leaderboard results are not available here.')
print('Decoding control (no-repeat token n-gram):', NO_REPEAT_NGRAM)